# Notebook Nro 8 - variantes en todas las etiquetas

## Idea de agrupamiento por etiqueta

No todas las etiquetas se comparan igual.  

| Etiqueta    | Cómo conviene agrupar variantes |
| ----------- | ------------------------------- |
| `persona`   | Por similitud de nombres        |
| `DNI`       | Por dígitos normalizados        |
| `CUIT_CUIL` | Por dígitos normalizados        |
| `CBU`       | Por dígitos normalizados        |
| `CVU`       | Por dígitos normalizados        |
| `ALIAS`     | Por texto normalizado           |
| `MONTO`     | Por valor numérico normalizado  |


# Celda 1 — Instalar dependencia para similitud

Vamos a usar rapidfuzz, que sirve para comparar textos parecidos.

In [17]:
!pip install -q rapidfuzz


## Celda 2 — Imports y explicación

In [18]:
# =========================
# NOTEBOOK 8
# Agrupar variantes de entidad persona
# =========================
#
# Objetivo:
# Crear un JSON/CSV donde las entidades persona estén agrupadas por persona probable.
#
# Ejemplo:
# PETRONE Maria Teresa
# María Taresa Petrone
# PETRONE Marta Teresa
#
# quedan dentro del mismo grupo de variantes.

import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz

# Celda 3 — Subir archivos

In [19]:
from google.colab import files

uploaded = files.upload()

print("Archivos subidos:")
for filename in uploaded.keys():
    print("-", filename)

Saving embargos_revision_entidades_actualizado_metodo_corregido.csv to embargos_revision_entidades_actualizado_metodo_corregido (1).csv
Saving embargos_revision_entidades_actualizado_metodo_corregido.json to embargos_revision_entidades_actualizado_metodo_corregido (1).json
Archivos subidos:
- embargos_revision_entidades_actualizado_metodo_corregido (1).csv
- embargos_revision_entidades_actualizado_metodo_corregido (1).json


# Celda 4 — Detectar archivos automáticamente

In [20]:
uploaded_files = list(uploaded.keys())

json_files = [f for f in uploaded_files if f.lower().endswith(".json")]
csv_files = [f for f in uploaded_files if f.lower().endswith(".csv")]

if len(json_files) != 1:
    raise ValueError(f"Se esperaba 1 JSON, encontrados: {json_files}")

if len(csv_files) != 1:
    raise ValueError(f"Se esperaba 1 CSV, encontrados: {csv_files}")

JSON_PATH = Path(json_files[0])
CSV_PATH = Path(csv_files[0])

print("JSON detectado:", JSON_PATH)
print("CSV detectado:", CSV_PATH)

JSON detectado: embargos_revision_entidades_actualizado_metodo_corregido (1).json
CSV detectado: embargos_revision_entidades_actualizado_metodo_corregido (1).csv


# Celda 5 — Configurar salidas

In [21]:
OUTPUT_JSON = Path("/content/embargos_personas_agrupadas_variantes.json")
OUTPUT_CSV = Path("/content/embargos_personas_agrupadas_variantes.csv")
OUTPUT_EXCEL = Path("/content/embargos_personas_agrupadas_variantes.xlsx")

print("Salida JSON:", OUTPUT_JSON)
print("Salida CSV:", OUTPUT_CSV)
print("Salida Excel:", OUTPUT_EXCEL)

Salida JSON: /content/embargos_personas_agrupadas_variantes.json
Salida CSV: /content/embargos_personas_agrupadas_variantes.csv
Salida Excel: /content/embargos_personas_agrupadas_variantes.xlsx


# Celda 5 — Cargar JSON y CSV

In [22]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.read_csv(CSV_PATH)

print("Documentos en JSON:", len(data))
print("Filas en CSV:", len(df))
print("Columnas CSV:")
print(df.columns.tolist())

Documentos en JSON: 80
Filas en CSV: 1626
Columnas CSV:
['numero_archivo', 'id', 'nombre_archivo', 'clasificacion', 'texto_limpio', 'cantidad_entidades_encontradas', 'etiqueta', 'valor', 'metodo', 'span_inicio', 'span_fin']


# Celda 6 — Cargar datos

In [23]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.read_csv(CSV_PATH)

print("Documentos cargados:", len(data))
print("Filas CSV:", len(df))

Documentos cargados: 80
Filas CSV: 1626


# Celda 7 — Funciones de normalización

In [24]:
import re
import unicodedata
from rapidfuzz import fuzz

def clean_value(x):
    if x is None:
        return None

    try:
        if pd.isna(x):
            return None
    except Exception:
        pass

    s = str(x).strip()
    return s if s else None


def clean_int_or_none(x):
    if x is None:
        return None

    try:
        if pd.isna(x):
            return None
    except Exception:
        pass

    try:
        return int(float(x))
    except Exception:
        return None


def quitar_tildes(texto):
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(ch for ch in texto if unicodedata.category(ch) != "Mn")
    return texto


def normalizar_etiqueta(etiqueta):
    etiqueta = clean_value(etiqueta)

    if etiqueta is None:
        return None

    etiqueta_low = etiqueta.strip().lower()

    mapping = {
        "persona": "persona",
        "dni": "DNI",
        "cuit": "CUIT_CUIL",
        "cuil": "CUIT_CUIL",
        "cuit_cuil": "CUIT_CUIL",
        "cuil_cuit": "CUIT_CUIL",
        "cbu": "CBU",
        "cvu": "CVU",
        "alias": "ALIAS",
        "monto": "MONTO",
    }

    return mapping.get(etiqueta_low, etiqueta)


def normalizar_metodo(metodo):
    metodo = clean_value(metodo)

    if metodo is None:
        return None

    mapping = {
        "Manual": "manual",
        "MANUAL": "manual",
        "Corregido": "corregido",
        "CORREGIDO": "corregido",
        "revision manual": "corregido",
        "revisión manual": "corregido",
    }

    return mapping.get(metodo, metodo)


def solo_digitos(valor):
    valor = clean_value(valor)
    if valor is None:
        return ""
    return re.sub(r"\D", "", valor)


def normalizar_texto_simple(valor):
    valor = clean_value(valor)
    if valor is None:
        return ""

    valor = quitar_tildes(valor)
    valor = valor.upper()
    valor = re.sub(r"[^A-Z0-9Ñ\s]", " ", valor)
    valor = re.sub(r"\s+", " ", valor).strip()

    return valor


def normalizar_nombre_persona(nombre):
    """
    Normaliza nombres para comparar variantes.
    No elimina títulos/roles como Dr, Dra, Juez, Secretario, etc.
    porque pueden servir luego para análisis de roles.
    """
    return normalizar_texto_simple(nombre)


def tokens_nombre(nombre):
    normalizado = normalizar_nombre_persona(nombre)

    return [
        t for t in normalizado.split()
        if len(t) >= 2
    ]


def normalizar_monto(valor):
    """
    Convierte montos argentinos a una clave comparable.

    Ejemplos:
    $ 108.332,50 -> 108332.50
    108332,50 -> 108332.50
    $ 81.249 -> 81249
    """
    valor = clean_value(valor)

    if valor is None:
        return None

    s = valor.strip()

    # Quitar símbolos de moneda y letras
    s = s.replace("$", "")
    s = s.replace("ARS", "")
    s = s.replace("pesos", "")
    s = s.replace("PESOS", "")
    s = s.strip()

    # Dejar solo números, puntos y comas
    s = re.sub(r"[^0-9\.,]", "", s)

    if not s:
        return None

    # Caso argentino: punto miles, coma decimal
    if "," in s:
        s = s.replace(".", "")
        s = s.replace(",", ".")
    else:
        # Si no hay coma, asumimos que los puntos son miles
        partes = s.split(".")
        if len(partes) > 1:
            s = "".join(partes)

    try:
        num = float(s)

        # Normalizar sin ceros innecesarios
        if num.is_integer():
            return str(int(num))

        return f"{num:.2f}"
    except Exception:
        return s


# Celda 8 — Función para decidir si dos nombres son variantes

In [25]:
# =========================
# FUNCIONES DE COMPARACIÓN DE VARIANTES
# Para agrupar variantes en todas las etiquetas
# =========================

def son_variantes_persona(valor_a, valor_b):
    """
    Decide si dos nombres probablemente representan la misma persona.

    Cubre casos:
    - mismo nombre exacto normalizado
    - mismo nombre en distinto orden
    - nombre completo vs nombre parcial
    - pequeñas diferencias OCR
    """
    norm_a = normalizar_nombre_persona(valor_a)
    norm_b = normalizar_nombre_persona(valor_b)

    if not norm_a or not norm_b:
        return False

    if norm_a == norm_b:
        return True

    tokens_a = tokens_nombre(valor_a)
    tokens_b = tokens_nombre(valor_b)

    if not tokens_a or not tokens_b:
        return False

    set_a = set(tokens_a)
    set_b = set(tokens_b)

    # Mismas palabras en distinto orden
    if set_a == set_b:
        return True

    # Nombre parcial contenido en nombre completo
    menor = set_a if len(set_a) <= len(set_b) else set_b
    mayor = set_b if len(set_a) <= len(set_b) else set_a

    if len(menor) >= 2 and menor.issubset(mayor):
        return True

    # Caso de una sola palabra, por ejemplo solo apellido.
    # Lo dejamos más estricto para no mezclar personas por error.
    if len(menor) == 1:
        token = list(menor)[0]

        if len(token) >= 5 and token in mayor:
            if fuzz.partial_ratio(norm_a, norm_b) >= 88:
                return True

    # Similitud por orden flexible
    token_sort = fuzz.token_sort_ratio(norm_a, norm_b)
    token_set = fuzz.token_set_ratio(norm_a, norm_b)

    if token_sort >= 82 or token_set >= 88:
        return True

    return False


def clave_comparacion_por_etiqueta(etiqueta, valor):
    """
    Devuelve una clave normalizada para comparar variantes según el tipo de entidad.
    """
    etiqueta = normalizar_etiqueta(etiqueta)
    valor = clean_value(valor)

    if valor is None:
        return ""

    # Entidades numéricas: se comparan por dígitos
    if etiqueta in ["DNI", "CUIT_CUIL", "CBU", "CVU"]:
        return solo_digitos(valor)

    # Montos: se comparan por valor numérico normalizado
    if etiqueta == "MONTO":
        return normalizar_monto(valor) or ""

    # Alias: se compara por texto normalizado
    if etiqueta == "ALIAS":
        return normalizar_texto_simple(valor)

    # Persona: se compara por nombre normalizado
    if etiqueta == "persona":
        return normalizar_nombre_persona(valor)

    return normalizar_texto_simple(valor)


def son_variantes_misma_entidad(ent_a, ent_b):
    """
    Decide si dos entidades son variantes de la misma entidad.

    Reglas:
    - Si son personas, usa similitud de nombres.
    - Si son DNI/CUIT/CBU/CVU, compara dígitos normalizados.
    - Si son montos, compara valor numérico normalizado.
    - Si son alias u otras etiquetas, compara texto normalizado.
    """
    etiqueta_a = normalizar_etiqueta(ent_a.get("etiqueta"))
    etiqueta_b = normalizar_etiqueta(ent_b.get("etiqueta"))

    if etiqueta_a != etiqueta_b:
        return False

    valor_a = clean_value(ent_a.get("valor"))
    valor_b = clean_value(ent_b.get("valor"))

    if valor_a is None or valor_b is None:
        return False

    # Persona usa similitud de nombres
    if etiqueta_a == "persona":
        return son_variantes_persona(valor_a, valor_b)

    # Regex y demás entidades usan clave normalizada exacta
    clave_a = clave_comparacion_por_etiqueta(etiqueta_a, valor_a)
    clave_b = clave_comparacion_por_etiqueta(etiqueta_b, valor_b)

    if not clave_a or not clave_b:
        return False

    return clave_a == clave_b

# Celda 9 — Agrupar personas dentro de cada archivo

Esta es la celda importante.

In [26]:
def agrupar_variantes_por_entidad(entidades):
    """
    Agrupa entidades de un mismo archivo por etiqueta y por valor equivalente.

    - persona: similitud de nombres.
    - DNI/CUIT/CBU/CVU: dígitos normalizados.
    - MONTO: valor numérico normalizado.
    - ALIAS: texto normalizado.
    """
    grupos = []

    for ent in entidades:
        etiqueta = normalizar_etiqueta(ent.get("etiqueta"))
        valor = clean_value(ent.get("valor"))

        if etiqueta is None or valor is None:
            continue

        ent_limpia = {
            "etiqueta": etiqueta,
            "valor": valor,
            "metodo": normalizar_metodo(ent.get("metodo")),
            "span_inicio": clean_int_or_none(ent.get("span_inicio")),
            "span_fin": clean_int_or_none(ent.get("span_fin")),
        }

        agregado = False

        for grupo in grupos:
            # Solo comparar contra grupos de la misma etiqueta
            if grupo["etiqueta"] != etiqueta:
                continue

            if any(
                son_variantes_misma_entidad(ent_limpia, variante)
                for variante in grupo["variantes"]
            ):
                grupo["variantes"].append(ent_limpia)
                agregado = True
                break

        if not agregado:
            grupos.append({
                "etiqueta": etiqueta,
                "variantes": [ent_limpia]
            })

    return grupos

# Celda 10 — Crear JSON correcto con grupos de variantes

In [27]:
# =========================
# CREAR JSON CON VARIANTES EN TODAS LAS ETIQUETAS
# =========================

json_entidades_variantes = []

for doc in data:
    numero_archivo = int(doc["numero_archivo"])

    entidades_originales = doc.get("entidades", [])

    grupos = agrupar_variantes_por_entidad(entidades_originales)

    entidades_finales = []

    contador_por_etiqueta = {}

    for grupo in grupos:
        etiqueta = grupo["etiqueta"]

        contador_por_etiqueta[etiqueta] = contador_por_etiqueta.get(etiqueta, 0) + 1
        idx_etiqueta = contador_por_etiqueta[etiqueta]

        variantes_finales = []

        for idx_variante, variante in enumerate(grupo["variantes"], start=1):
            variantes_finales.append({
                "id_variante": str(idx_variante),
                "valor": variante["valor"],
                "metodo": variante["metodo"],
                "span_inicio": variante["span_inicio"],
                "span_fin": variante["span_fin"],
            })

        entidades_finales.append({
            "id_etiqueta": f"{numero_archivo:03d}_{etiqueta}_{idx_etiqueta:03d}",
            "etiqueta": etiqueta,
            "variantes": variantes_finales,
        })

    nuevo_doc = {
        "numero_archivo": numero_archivo,
        "id": doc.get("id"),
        "nombre_archivo": doc.get("nombre_archivo"),
        "clasificacion": doc.get("clasificacion"),
        "texto_limpio": doc.get("texto_limpio"),
        "entidades": entidades_finales,
    }

    json_entidades_variantes.append(nuevo_doc)

print("Documentos procesados:", len(json_entidades_variantes))

print("\nEjemplo primer documento:")
print(json.dumps(json_entidades_variantes[0], ensure_ascii=False, indent=2)[:5000])

Documentos procesados: 80

Ejemplo primer documento:
{
  "numero_archivo": 1,
  "id": "538118",
  "nombre_archivo": "Embargo - usuario",
  "clasificacion": "Embargo",
  "texto_limpio": "18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000800 000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr\n\n| Orgañismo: juzgado EN LO CIVIL Y COMERCIAL N0%16 - SA\n| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO\nNúmero de causa: s/prrse-2093 o\nTipo de notificación: Jeeps ELECTRONICO 4 paar\nDestinatarios: A Y\nFécha notificación: 20/2/2026 \" O.\nAlta a disponibilidad 19/2/2026 10:44:34\n| Fifmal digital: Firma valida\nFirmablo y Notificado por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026\n10:44.33\nFifmabo por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026\n\n2]¡BRANDARIZ Luciana Rocio. --- Certificado Correcto. Fecha de Firma:\n2:20p6 32:44.42 THEILER Pa

# Celda 11 — Guardar JSON

In [28]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(json_entidades_variantes, f, ensure_ascii=False, indent=2)

print("JSON guardado:", OUTPUT_JSON)

JSON guardado: /content/embargos_personas_agrupadas_variantes.json


# Celda 12 — Crear CSV aplanado

Cada fila es una variante, pero ahora con su grupo/persona probable.

In [29]:
rows = []

for doc in json_entidades_variantes:
    for entidad in doc["entidades"]:
        for variante in entidad["variantes"]:
            rows.append({
                "numero_archivo": doc["numero_archivo"],
                "id": doc["id"],
                "nombre_archivo": doc["nombre_archivo"],
                "clasificacion": doc["clasificacion"],
                "texto_limpio": doc["texto_limpio"],

                "id_etiqueta": entidad["id_etiqueta"],
                "etiqueta": entidad["etiqueta"],

                "id_variante": variante["id_variante"],
                "valor": variante["valor"],
                "metodo": variante["metodo"],
                "span_inicio": variante["span_inicio"],
                "span_fin": variante["span_fin"],
            })

df_entidades_variantes = pd.DataFrame(rows)

df_entidades_variantes.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("CSV guardado:", OUTPUT_CSV)
print("Total variantes:", len(df_entidades_variantes))
print("Total grupos entidad:", df_entidades_variantes["id_etiqueta"].nunique())

df_entidades_variantes.head()

CSV guardado: /content/embargos_personas_agrupadas_variantes.csv
Total variantes: 1622
Total grupos entidad: 1220


,numero_archivo,id,nombre_archivo,clasificacion,texto_limpio,id_etiqueta,etiqueta,id_variante,valor,metodo,span_inicio,span_fin
0,1,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓ...,001_DNI_001,DNI,1,33.470.065,regex,1710,1720
1,1,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓ...,001_MONTO_001,MONTO,1,"$ 108.332,50",regex_contextual,1951,1962
2,1,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓ...,001_MONTO_001,MONTO,2,"$ 108.332,50",regex_contextual,4788,4799
3,1,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓ...,001_MONTO_002,MONTO,1,$ 81.249,regex_contextual,2083,2090
4,1,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓ...,001_MONTO_002,MONTO,2,$ 81.249,regex_contextual,4928,4935


# Validación para saber si encontró variantes

In [30]:
# =========================
# VALIDAR GRUPOS CON MÁS DE UNA VARIANTE
# =========================

grupos_variantes = (
    df_entidades_variantes
    .groupby(["numero_archivo", "id_etiqueta", "etiqueta"])
    .size()
    .reset_index(name="cantidad_variantes")
)

print("Total grupos entidad:", len(grupos_variantes))

print("\nGrupos con más de una variante:")
grupos_con_variantes = grupos_variantes[grupos_variantes["cantidad_variantes"] > 1]
print(grupos_con_variantes.sort_values("cantidad_variantes", ascending=False).head(30))

print("\nCantidad de grupos con variantes por etiqueta:")
print(
    grupos_con_variantes["etiqueta"]
    .value_counts()
)

print("\nPromedio de variantes por etiqueta:")
print(
    grupos_variantes
    .groupby("etiqueta")["cantidad_variantes"]
    .mean()
    .sort_values(ascending=False)
)

Total grupos entidad: 1220

Grupos con más de una variante:
      numero_archivo        id_etiqueta   etiqueta  cantidad_variantes
359               27    027_persona_001    persona                   5
236               18      018_MONTO_001      MONTO                   5
1146              75    075_persona_002    persona                   5
1096              72    072_persona_002    persona                   5
422               32    032_persona_001    persona                   5
11                 1    001_persona_002    persona                   4
698               46      046_MONTO_001      MONTO                   4
118               10    010_persona_002    persona                   4
1030              67    067_persona_001    persona                   4
1029              67      067_MONTO_001      MONTO                   4
1104              73      073_MONTO_002      MONTO                   4
423               32    032_persona_002    persona                   4
46               

# Celda 13 — Excel de revisión

In [31]:
# =========================
# CREAR EXCEL DE REVISIÓN
# Todas las entidades con variantes
# =========================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils import get_column_letter

df_excel = df_entidades_variantes.copy()

# texto_limpio solo en la primera fila de cada archivo
if not df_excel.empty:
    df_excel["texto_limpio"] = df_excel.groupby("numero_archivo")["texto_limpio"].transform(
        lambda s: [str(s.iloc[0])[:1200]] + [""] * (len(s) - 1)
    )

df_excel.to_excel(OUTPUT_EXCEL, index=False, sheet_name="Entidades_Variantes")

wb = load_workbook(OUTPUT_EXCEL)
ws = wb["Entidades_Variantes"]

header_fill = PatternFill("solid", fgColor="17365D")
header_font = Font(color="FFFFFF", bold=True)
thin = Side(style="thin", color="D9E2F3")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

fill_odd = PatternFill("solid", fgColor="EAF3FF")
fill_even = PatternFill("solid", fgColor="FFF2CC")
fill_group_odd = PatternFill("solid", fgColor="9DC3E6")
fill_group_even = PatternFill("solid", fgColor="FFD966")

# Header
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = border

headers = {cell.value: cell.column for cell in ws[1]}

numero_col = headers.get("numero_archivo")
id_etiqueta_col = headers.get("id_etiqueta")
etiqueta_col = headers.get("etiqueta")

# Body
for row_idx in range(2, ws.max_row + 1):
    numero = ws.cell(row=row_idx, column=numero_col).value

    try:
        numero_int = int(numero)
    except Exception:
        numero_int = 0

    fill = fill_odd if numero_int % 2 == 1 else fill_even
    fill_group = fill_group_odd if numero_int % 2 == 1 else fill_group_even

    for col_idx in range(1, ws.max_column + 1):
        cell = ws.cell(row=row_idx, column=col_idx)
        cell.fill = fill
        cell.border = border
        cell.alignment = Alignment(vertical="top", wrap_text=True)

    # Resaltar número de archivo
    if numero_col:
        ws.cell(row=row_idx, column=numero_col).fill = fill_group
        ws.cell(row=row_idx, column=numero_col).font = Font(bold=True)

    # Resaltar grupo de entidad
    if id_etiqueta_col:
        ws.cell(row=row_idx, column=id_etiqueta_col).font = Font(bold=True)

    # Resaltar etiqueta
    if etiqueta_col:
        ws.cell(row=row_idx, column=etiqueta_col).font = Font(bold=True)

# Anchos de columnas
widths = {
    "numero_archivo": 16,
    "id": 14,
    "nombre_archivo": 26,
    "clasificacion": 18,
    "texto_limpio": 80,
    "id_etiqueta": 26,
    "etiqueta": 16,
    "id_variante": 14,
    "valor": 40,
    "metodo": 26,
    "span_inicio": 14,
    "span_fin": 14,
}

for header, width in widths.items():
    if header in headers:
        ws.column_dimensions[get_column_letter(headers[header])].width = width

ws.freeze_panes = "A2"
ws.auto_filter.ref = ws.dimensions

# =========================
# HOJA RESUMEN
# =========================

ws_resumen = wb.create_sheet("Resumen")

ws_resumen.append(["Métrica", "Valor"])
ws_resumen.append(["Documentos", len(json_entidades_variantes)])
ws_resumen.append(["Total grupos entidad", df_entidades_variantes["id_etiqueta"].nunique()])
ws_resumen.append(["Total variantes", len(df_entidades_variantes)])
ws_resumen.append(["Etiquetas distintas", df_entidades_variantes["etiqueta"].nunique()])

ws_resumen.append([])
ws_resumen.append(["Variantes por etiqueta", "Cantidad"])

for etiqueta, cantidad in df_entidades_variantes["etiqueta"].value_counts().items():
    ws_resumen.append([etiqueta, int(cantidad)])

ws_resumen.append([])
ws_resumen.append(["Grupos por etiqueta", "Cantidad"])

grupos_por_etiqueta = (
    df_entidades_variantes
    .drop_duplicates(["id_etiqueta"])
    ["etiqueta"]
    .value_counts()
)

for etiqueta, cantidad in grupos_por_etiqueta.items():
    ws_resumen.append([etiqueta, int(cantidad)])

ws_resumen.append([])
ws_resumen.append(["Variantes por método", "Cantidad"])

for metodo, cantidad in df_entidades_variantes["metodo"].value_counts().items():
    ws_resumen.append([metodo, int(cantidad)])

for cell in ws_resumen[1]:
    cell.fill = header_fill
    cell.font = header_font

ws_resumen.column_dimensions["A"].width = 34
ws_resumen.column_dimensions["B"].width = 18

wb.save(OUTPUT_EXCEL)

print("Excel guardado:", OUTPUT_EXCEL)

Excel guardado: /content/embargos_personas_agrupadas_variantes.xlsx


# Celda 14 — Validaciones

In [32]:
# =========================
# VALIDACIONES FINALES
# Todas las etiquetas con variantes
# =========================

print("Documentos:", len(json_entidades_variantes))
print("Total grupos entidad:", df_entidades_variantes["id_etiqueta"].nunique())
print("Total variantes:", len(df_entidades_variantes))
print("Etiquetas distintas:", df_entidades_variantes["etiqueta"].nunique())

print("\nVariantes por etiqueta:")
print(
    df_entidades_variantes["etiqueta"]
    .value_counts()
)

print("\nGrupos por etiqueta:")
print(
    df_entidades_variantes
    .drop_duplicates(["id_etiqueta"])
    ["etiqueta"]
    .value_counts()
)

print("\nVariantes por método:")
print(
    df_entidades_variantes["metodo"]
    .value_counts()
)

print("\nPromedio de grupos entidad por archivo:")
print(
    df_entidades_variantes
    .groupby("numero_archivo")["id_etiqueta"]
    .nunique()
    .mean()
)

print("\nPromedio de variantes por grupo entidad:")
print(
    df_entidades_variantes
    .groupby("id_etiqueta")
    .size()
    .mean()
)

print("\nTop 20 grupos con más variantes:")
print(
    df_entidades_variantes
    .groupby(["numero_archivo", "id_etiqueta", "etiqueta"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

print("\nGrupos con más de una variante por etiqueta:")

grupos_variantes = (
    df_entidades_variantes
    .groupby(["numero_archivo", "id_etiqueta", "etiqueta"])
    .size()
    .reset_index(name="cantidad_variantes")
)

grupos_con_mas_de_una_variante = grupos_variantes[
    grupos_variantes["cantidad_variantes"] > 1
]

print(
    grupos_con_mas_de_una_variante["etiqueta"]
    .value_counts()
)

print("\nCantidad total de grupos con más de una variante:")
print(len(grupos_con_mas_de_una_variante))

print("\nTop 20 grupos con más de una variante:")
print(
    grupos_con_mas_de_una_variante
    .sort_values("cantidad_variantes", ascending=False)
    .head(20)
)

Documentos: 80
Total grupos entidad: 1220
Total variantes: 1622
Etiquetas distintas: 7

Variantes por etiqueta:
etiqueta
persona      1031
MONTO         255
DNI           163
CUIT_CUIL      91
CBU            77
CVU             4
ALIAS           1
Name: count, dtype: int64

Grupos por etiqueta:
etiqueta
persona      784
MONTO        156
DNI          136
CUIT_CUIL     75
CBU           65
CVU            3
ALIAS          1
Name: count, dtype: int64

Variantes por método:
metodo
gliner_fastino             911
regex                      336
regex_contextual           255
corregido                  105
regex_persona_cerca_dni     15
Name: count, dtype: int64

Promedio de grupos entidad por archivo:
15.25

Promedio de variantes por grupo entidad:
1.3295081967213114

Top 20 grupos con más variantes:
numero_archivo  id_etiqueta      etiqueta
18              018_MONTO_001    MONTO       5
32              032_persona_001  persona     5
75              075_persona_002  persona     5
72             

# Celda 15 — Descargar archivos

In [33]:
from google.colab import files

files.download(str(OUTPUT_JSON))
files.download(str(OUTPUT_CSV))
files.download(str(OUTPUT_EXCEL))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>